# 09 — Run the complete forecasting experiment without Microsoft Fabric

This notebook runs the **real-data local pipeline** through Python, DuckDB and scikit-learn. It downloads source data directly from the official NYC TLC host. **No performance results are embedded or invented**: outputs appear only after successful execution. A local Jupyter notebook or VS Code kernel is required. Source files may take some time to download.

**Protocol:** January 8–20 training; January 21–25 validation; January 26–31 retrospective, previously inspected test. The predictions are one-hour-ahead assuming the previous actual hourly count is known.

In [ ]:
from pathlib import Path
import os

project = Path.cwd()
if not (project / "scripts" / "download_data.py").exists():
    project = project.parent
assert (project / "scripts" / "run_local.py").exists(), "Open Jupyter from the repository root or notebooks/ folder"
print("Project root:", project)

## Download official TLC raw data

Run the project requirements installation in a terminal first: `pip install -r requirements-local.txt`. Source Parquet and zone lookup are supplied by NYC TLC, not this project.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(project / "scripts" / "download_data.py")], check=True, cwd=str(project))

## Execute Silver, Gold, feature engineering and three ML candidates

This launches the DuckDB/Scikit-learn pipeline, **not Fabric**. The result filenames are fixed so that a new execution updates rather than silently accumulates stale reports.

In [ ]:
subprocess.run([sys.executable, str(project / "scripts" / "run_local.py")], check=True, cwd=str(project))

## Load the generated machine-learning report

Do not commit a report under a claim of independent test validation: January 26–31 had already been inspected in prior project work.

In [ ]:
from IPython.display import Markdown, display, Image
reports = project / "results" / "local_run"
display(Markdown((reports / "EXECUTED_REPORT.md").read_text()))
display(Image(filename=str(reports / "validation_model_comparison.png")))

In [ ]:
import pandas as pd
comparison = pd.read_csv(reports / "validation_model_comparison.csv")
category = pd.read_csv(reports / "validation_category_metrics.csv")
display(comparison)
display(category)

## Publication checklist

- Verify row and pickup reconciliations with your original Fabric observations.
- Review metric values and category-specific error, not only the headline MAE.
- Include the generated report and charts as **executed results** after running.
- Add further months for a genuinely unseen out-of-time evaluation.
- Update the README's execution status only after successful source-data execution.